In [27]:
import pandas as pd
from pathlib import Path

from pydantic import BaseModel

import os

import constants

from prompting_utils import create_system_prompt

from model_utils import from_series_to_basemodel

from tqdm import tqdm

from anthropic import Anthropic

from pathlib import Path

import json

# Preliminari

In [ ]:
# Configurazione OpenAI
client = Anthropic(
    api_key=os.getenv("ANTHROPIC_API_KEY")
)

# Parametri
base_dir = Path.cwd().parent
SYSTEM_PROMPT_FILE_NAME = constants.SYSTEM_PROMPT_4
TEMPERATURE = 0.0

MODEL = constants.CLAUDE_OPUS_4_6
RESULTS_FILE_NAME = 'results_' + 'opus-4.6' + '.jsonl'

PYDANTIC_MODEL = constants.RectalCancerStagingData

#Carica system prompt da file
system_prompt_path = base_dir / "data" / "prompts" / SYSTEM_PROMPT_FILE_NAME
system_prompt = create_system_prompt(system_prompt_path, PYDANTIC_MODEL)

In [3]:
if True:
    print(system_prompt)

Sei un assistente medico radiologico esperto nella stadiazione del carcinoma rettale tramite RM.

Il tuo compito è estrarre informazioni strutturate dal referto RM fornito e restituire esclusivamente un oggetto JSON valido conforme allo schema seguente:

{"morfologia": "solido_polipoide | solido_anulare | mucinoso", "ore_inizio": "int | null", "ore_fine": "int | null", "spessore_parietale": "int | null", "estensione_cranio_caudale": "int | null", "distanza_oai": "int | null", "posizione": {"basso": "no | si", "medio": "no | si", "alto": "no | si", "giunzione": "no | si"}, "riflessione_peritoneale_anteriore": "sotto | cavallo | non_valutabile", "infiltrazione_tessuto_adiposo": "no | si_5mm | si_5mm_plus", "infiltrazione_sfinteri": "no | si", "infiltrazione_organi_extra": "no | si", "infiltrazione_organi_dettagli": {"pavimento_pelvico": "no | si", "altro": "no | si"}, "coinvolgimento_riflessione_peritoneale": "no | si", "coinvolgimento_fascia_mesorettale": "no | si", "numero_linfonodi_no

# Load Data

In [4]:
# Carichiamo i nostri file csv
file_names = {
    'validation': constants.VALIDATION_SPLIT_FILE_NAME,
    'test': constants.TEST_SPLIT_FILE_NAME,
}

paths = {split: Path('../data/' + file_name) for split, file_name in file_names.items()}

data = dict()
for split, path in paths.items():
    data[split] = pd.read_csv(path)

validation_data, test_data = data['validation'], data['test']

################################
# Convert float columns to Int64
################################
float_cols = test_data.select_dtypes("float").columns
for col in float_cols:
    test_data[col] = test_data[col].round().astype("Int64")
    validation_data[col] = validation_data[col].round().astype("Int64")
    
# Check duplicatest
assert set(test_data.id) & set(validation_data.id) == set(), "There are overlapping IDs between test and validation sets!"

print(f"{len(test_data) = }")
print(f"{len(validation_data) = }")

len(test_data) = 65
len(validation_data) = 66


# Generazione

## Preliminari generazione

In [19]:
# Funzione generatrice
def extract_data_from_report(model: str,
                             report_text: str,
                             system_prompt: str = system_prompt,
                             temperature: float = TEMPERATURE,
                             output_format: type[BaseModel] = PYDANTIC_MODEL):
    response = client.messages.parse(
        model=model,
        max_tokens=1024,
        system=system_prompt,
        temperature=temperature,
        messages=[
            {'role': 'user', 'content': report_text}
        ],
        output_format=output_format
    )
    return response

In [20]:
print(MODEL)

claude-opus-4-6


In [21]:
# Esempio
if True:
    result = extract_data_from_report(MODEL, data['test'].iloc[0]['report_text'])

In [22]:
if False: # To see the full output
    pprint(result.model_dump())
if True:  # To see the string output text
    print(result.content[0].text)
if True:  # To see the parsed output as a pydantic BaseModel
    print(result.content[0].parsed_output)
if True:
    print(result.content[0].parsed_output.model_dump(mode='json'))

{"morfologia":"solido_polipoide","ore_inizio":3,"ore_fine":9,"spessore_parietale":null,"estensione_cranio_caudale":null,"distanza_oai":null,"posizione":{"basso":"si","medio":"no","alto":"no","giunzione":"no"},"riflessione_peritoneale_anteriore":"sotto","infiltrazione_tessuto_adiposo":"no","infiltrazione_sfinteri":"si","infiltrazione_organi_extra":"si","infiltrazione_organi_dettagli":{"pavimento_pelvico":"si","altro":"no"},"coinvolgimento_riflessione_peritoneale":"no","coinvolgimento_fascia_mesorettale":"no","numero_linfonodi_non_conosciuto":"conosciuto","linfonodi_sospetti":1,"sedi_linfonodi":{"mesorettali":"no","rettali_superiori":"no","otturatori":"no","iliaci":"no","altro":"si"},"depositi_tumorali":"si","emvi":"+","stadio_T":"T4b","stadio_N":"N1","stadio_N1c":"si","mrf":"-","metastasi":"MX"}
morfologia='solido_polipoide' ore_inizio=3 ore_fine=9 spessore_parietale=None estensione_cranio_caudale=None distanza_oai=None posizione=PosizioneFlags(basso='si', medio='no', alto='no', giunzio

## Baseline inference
Usiamo modelli non finetunati. Solo prompt engineering.

In [ ]:
print(MODEL)
df = pd.concat([validation_data, test_data], ignore_index=True)
ids = []
actual = []
predicted = []
splits = []
for i in tqdm(range(df.shape[0])):
    row = df.iloc[i]
    splits.append(row['split'])
    output = extract_data_from_report(model=MODEL, report_text=row[constants.REPORT_COLUMN_NAME])
    real = from_series_to_basemodel(row, PYDANTIC_MODEL)
    ids.append(row[constants.ID_COMULM_NAME])
    actual.append(real.model_dump(mode='json'))
    if output is None:
        predicted.append('no output')
    else:
        predicted.append(output.content[0].parsed_output.model_dump(mode='json'))

claude-opus-4-6


100%|██████████| 12/12 [00:57<00:00,  4.82s/it]


In [28]:
results_dicts = []
assert len(ids) == len(actual) == len(predicted)
for id, act, pred, split in zip(ids, actual, predicted, splits):
    results_dicts.append(
        {
            'model': MODEL,
            'split': split,
            'id': int(id),
            'actual': act,
            'prediction': pred
        }
    )
# Salvataggio
with open(base_dir / 'data' / 'inference' / RESULTS_FILE_NAME, 'w', encoding='utf-8') as f:
    for r in results_dicts:
        f.write(json.dumps(r) + '\n')

In [29]:
output.model_dump()

c:\Users\lucat\PythonRepositories\PRIN\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed_output', input_value=RectalCancerStagingData(m...mrf='-', metastasi='MX'), input_type=RectalCancerStagingData])
  PydanticSerializationUnexpectedValue(Expected `ThinkingBlock` - serialized value may not be as expected [field_name='content', input_value=ParsedTextBlock[~Response...rf='-', metastasi='MX')), input_type=ParsedTextBlock[~ResponseFormatT]])
  PydanticSerializationUnexpectedValue(Expected `RedactedThinkingBlock` - serialized value may not be as expected [field_name='content', input_value=ParsedTextBlock[~Response...rf='-', metastasi='MX')), input_type=ParsedTextBlock[~ResponseFormatT]])
  PydanticSerializationUnexpectedValue(Expected `ToolUseBlock` - serialized value may not be as expected [field_name='content', input_value=ParsedTextB

{'id': 'msg_019bxqvymLYNHto1NZpLYUM8',
 'content': [{'citations': None,
   'text': '{"morfologia":"solido_anulare","ore_inizio":null,"ore_fine":null,"spessore_parietale":null,"estensione_cranio_caudale":40,"distanza_oai":50,"posizione":{"basso":"no","medio":"si","alto":"si","giunzione":"no"},"riflessione_peritoneale_anteriore":"cavallo","infiltrazione_tessuto_adiposo":"si_5mm_plus","infiltrazione_sfinteri":"no","infiltrazione_organi_extra":"no","infiltrazione_organi_dettagli":{"pavimento_pelvico":"no","altro":"no"},"coinvolgimento_riflessione_peritoneale":"si","coinvolgimento_fascia_mesorettale":"no","numero_linfonodi_non_conosciuto":"conosciuto","linfonodi_sospetti":2,"sedi_linfonodi":{"mesorettali":"no","rettali_superiori":"si","otturatori":"no","iliaci":"no","altro":"no"},"depositi_tumorali":"si","emvi":"+","stadio_T":"T4a","stadio_N":"N1","stadio_N1c":"si","mrf":"-","metastasi":"MX"}',
   'type': 'text',
   'parsed_output': {'morfologia': 'solido_anulare',
    'ore_inizio': None,
 